# Envelope-detector GR metrics — Diff-SSL-G-Comp (whole downloaded set)

Companion to [envelope_gr_comparison.ipynb](envelope_gr_comparison.ipynb). That notebook *visualised*
how different level / envelope detectors estimate the time-varying **gain reduction (GR)** and
reconstruct the wet signal as `pred_wet = dry · 10^(GR/20)`. This notebook **quantifies** every one of
those detectors over the **entire downloaded Diff-SSL-G-Comp dataset** (all settings × songs),
reporting **overall, duration-weighted** metrics only (no per-song rows).

For each detector the pipeline is identical to the comparison notebook:
`envelope → dB → GR → gain → reconstruction`. The metrics are grouped into two stages plus the
standard set used in
[eval_output_transformer_input_x_gr_curve.ipynb](../02c_sota_anal_final/eval_output_transformer_input_x_gr_curve.ipynb).

**Detectors** (same as the comparison notebook): Hilbert · RMS-64 · RMS-256 · RMS-1024 · RMS-4096 ·
Simple diff (per-sample `wet/dry`, the degenerate near-perfect-reconstruction anchor).

### Gain-reduction stage — the slow GR trajectory standard losses under-fit
| Metric | Measures | Source |
|---|---|---|
| **GR MAE (dB)** | mean \|GR_est − GR_ref\| on the trajectory | eval notebook |
| **MR-STE** | multi-resolution short-time **energy** envelope distance — shape, tolerant of small misalignments | Wright & Välimäki (grey-box) |
| **FoES** | Fidelity of Envelope Shape — normalised RMS error of the GR trajectory | Giannoulis et al. |
| **ESR** | strictly sample-aligned error-to-signal ratio (`pred_wet` vs `wet`) — the anchor that still exposes a systematic timing offset | — |

### Colour stage — residual timbre the smooth gain model can't explain
| Metric | Measures | Source |
|---|---|---|
| **MR-STFT (auraloss)** | multi-resolution STFT distance (spectral-convergence + log-magnitude) | auraloss / Steinmetz & Reiss, via `nablafx.evaluation` |
| **ESR (A-wt)** | A-weighted error-to-signal ratio — paper-comparable headline number | Wright & Välimäki |

### Standard (same as the SOTA eval notebook)
MAE (L1) · MSE (L2) · MR-STFT (`src.losses`) · EDC · M_NRMSE · M_SF.

**Notes**
- GR-trajectory metrics (GR MAE, FoES) compare against the reference RMS GR
  (`src.dsp_torch.gain_reduction_db`, window 1024) and clamp both curves to `[−30, 0] dB` exactly like
  the eval notebook; the **reconstruction** itself uses the *unclamped* GR (faithful to the comparison
  notebook).
- Heavy metrics run through a **batched, cumsum-accelerated** engine (bounded memory, ~20 s/pair →
  ~30 min for the full 100-pair set). The verification cell asserts the colour-stage MR-STFT / ESR
  reproduce `nablafx.evaluation` (auraloss) to < 1e-3.
- Set `MAX_EVAL_PAIRS` to a small int for a quick smoke test.


In [1]:
import os
import sys
import gc
import types
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from scipy.signal import hilbert, bilinear, lfilter
from tqdm.auto import tqdm

warnings.filterwarnings("ignore", category=UserWarning)

# Resolve repo root robustly (independent of the launch directory)
REPO_ROOT = next(
    (p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "pyproject.toml").is_file()),
    Path.cwd().resolve(),
)
# eval_helpers (dry/wet pairing + segment reader) lives in 03_initial_GR_pred,
# the same source the SOTA eval notebooks use.
sys.path.insert(0, str(REPO_ROOT / "03_initial_GR_pred"))

# nablafx's top-level __init__ eagerly imports processors.ddsp -> `rational`
# (an optional dependency that isn't installed here). Stub it so the pure
# auraloss/torch wrappers in `nablafx.evaluation` import cleanly.
for _name in ("rational", "rational.torch"):
    sys.modules.setdefault(_name, types.ModuleType(_name))
sys.modules["rational.torch"].Rational = object

from eval_helpers import (
    same_setting_pairs,
    _song_from_dry_path,
    _read_dry_wet_segment,
    _pair_num_frames,
)
from src.dsp import to_amplitude
from src.dsp_torch import GR_DB_MIN, GR_DB_MAX, RMS_WINDOW
from nablafx.evaluation import get_function, list_available_metrics

print(f"repo : {REPO_ROOT}")
print(f"torch: {torch.__version__}")
print(f"nablafx.evaluation metrics: {len(list_available_metrics())} registered "
      f"(using mrstft_loss, esr_loss)")

/Volumes/Saola's Drive/AllCode/thesis/Virtual-Analogue-Compressor-Modelling/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


repo : /Volumes/Saola's Drive/AllCode/thesis/Virtual-Analogue-Compressor-Modelling
torch: 2.10.0
nablafx.evaluation metrics: 25 registered (using mrstft_loss, esr_loss)


## Configuration


In [2]:
# Diff-SSL-G-Comp (SSL G-Bus hardware, 44.1 kHz)
DATA_ROOT   = "/Volumes/Saola's Drive/AllCode/thesis/data/Diff-SSL-G-Comp"
SAMPLE_RATE = 44100
EPS         = 1e-10

# Streaming chunking -> bounded memory; STEP alignment matches the SOTA eval notebooks
STEP             = 256 * 8                       # 2048
STREAM_CHUNK_SEC = 10.0
CHUNK_FRAMES     = int(round(STREAM_CHUNK_SEC * SAMPLE_RATE))
CHUNK_FRAMES    -= CHUNK_FRAMES % STEP

# Envelope detectors compared (exactly those in envelope_gr_comparison.ipynb)
RMS_WINDOWS = (64, 256, 1024, 4096)
DETECTORS   = ["Hilbert"] + [f"RMS {w}" for w in RMS_WINDOWS] + ["Simple diff"]

# Multi-resolution window / FFT sets
MR_STE_WINDOWS   = (256, 1024, 4096)            # short-time energy (envelope shape)
MR_NRMSE_WINDOWS = (512, 1024, 2048)            # == src.losses.multi_resolution_nrmse
STFT_SIZES       = (512, 1024, 2048)            # == src.losses.m_stfte / spectral-flux

# Quick smoke test: set to a small int (e.g. 2) to run on a few pairs only
MAX_EVAL_PAIRS = None

OUT_CSV = REPO_ROOT / "01b_dset_anal" / "envelope_metrics_diffssl.csv"

# Metric columns, grouped: GR-stage | colour-stage | standard (eval-notebook)
GR_STAGE_COLS     = ["GR MAE (dB)", "MR-STE", "FoES", "ESR"]
COLOUR_STAGE_COLS = ["MR-STFT (auraloss)", "ESR (A-wt)"]
STANDARD_COLS     = ["MAE (L1)", "MSE (L2)", "MR-STFT", "EDC", "M_NRMSE", "M_SF"]
METRIC_COLS       = GR_STAGE_COLS + COLOUR_STAGE_COLS + STANDARD_COLS

assert os.path.isdir(DATA_ROOT), f"Missing DATA_ROOT: {DATA_ROOT}"
print(f"chunk    : {STREAM_CHUNK_SEC}s ~ {CHUNK_FRAMES} frames, step={STEP}")
print(f"detectors: {DETECTORS}")
print(f"metrics  : {METRIC_COLS}")

chunk    : 10.0s ~ 440320 frames, step=2048
detectors: ['Hilbert', 'RMS 64', 'RMS 256', 'RMS 1024', 'RMS 4096', 'Simple diff']
metrics  : ['GR MAE (dB)', 'MR-STE', 'FoES', 'ESR', 'MR-STFT (auraloss)', 'ESR (A-wt)', 'MAE (L1)', 'MSE (L2)', 'MR-STFT', 'EDC', 'M_NRMSE', 'M_SF']


## Metric engine

All moving-average (RMS / energy) envelopes use a **prefix-sum (cumsum)** sliding window — `O(N)`
regardless of window length, validated to match a direct `conv1d` to ~1e-7. The FFT-based metrics are
computed **once per chunk, batched across all detectors** (the target STFT is shared), which is what
keeps the full run to ~30 min in bounded memory.


In [3]:
# ── Envelope / level helpers (cumsum moving-average: O(N), window-independent) ──
def _moving_meansq(x_sq: np.ndarray, W: int, centered: bool) -> np.ndarray:
    """Sliding mean of x**2 via prefix sums. centered=symmetric window, else causal (trailing)."""
    n = x_sq.shape[-1]
    c = np.empty(n + 1); c[0] = 0.0; np.cumsum(x_sq, out=c[1:])
    i = np.arange(n)
    if centered:
        lo = i - (W // 2); hi = lo + W
    else:                                       # causal window == src.dsp_torch.gain_reduction_db
        hi = i + 1; lo = hi - W
    lo = np.clip(lo, 0, n); hi = np.clip(hi, 0, n)
    return (c[hi] - c[lo]) / np.maximum(hi - lo, 1)

def _env_rms(x, W): return np.sqrt(_moving_meansq(x * x, W, centered=True))
def _to_db(x):      return 20.0 * np.log10(np.maximum(x, EPS))
def _esr(pred, tgt): return float(np.sum((tgt - pred) ** 2) / (np.sum(tgt * tgt) + 1e-8))

# A-weighting IIR (analog prototype -> bilinear) for the A-weighted ESR
def _a_weighting_ba(fs):
    f1, f2, f3, f4 = 20.598997, 107.65265, 737.86223, 12194.217
    A1000 = 1.9997
    nums = [(2 * np.pi * f4) ** 2 * 10 ** (A1000 / 20.0), 0, 0, 0, 0]
    dens = np.polymul([1, 4 * np.pi * f4, (2 * np.pi * f4) ** 2],
                      [1, 4 * np.pi * f1, (2 * np.pi * f1) ** 2])
    dens = np.polymul(np.polymul(dens, [1, 2 * np.pi * f3]), [1, 2 * np.pi * f2])
    return bilinear(nums, dens, fs)
AW_B, AW_A = _a_weighting_ba(SAMPLE_RATE)

# ── GR trajectories ──
def detector_gr_db(label, dry, wet):
    """Estimated gain-reduction trajectory (dB) for one envelope detector (unclamped)."""
    if label == "Hilbert":
        return _to_db(np.abs(hilbert(wet))) - _to_db(np.abs(hilbert(dry)))
    if label.startswith("RMS"):
        W = int(label.split()[1])
        return _to_db(_env_rms(wet, W)) - _to_db(_env_rms(dry, W))
    # Simple diff: degenerate per-sample gain wet/dry (perfect recon, noisy GR)
    gain = np.divide(wet, dry, out=np.ones_like(wet), where=np.abs(dry) > EPS)
    return _to_db(np.abs(gain))

def reference_gr_db(dry, wet, W=RMS_WINDOW):
    """Reference GR trajectory = causal RMS GR (matches src.dsp_torch.gain_reduction_db)."""
    return (_to_db(np.sqrt(_moving_meansq(wet * wet, W, centered=False)))
            - _to_db(np.sqrt(_moving_meansq(dry * dry, W, centered=False))))

# ── GR-stage metrics ──
def mr_ste(pred, wet, windows=MR_STE_WINDOWS):
    """Multi-resolution short-time energy distance (envelope shape; Wright & Valimaki)."""
    pe, we = pred * pred, wet * wet
    total = 0.0
    for W in windows:
        ep = _moving_meansq(pe, W, centered=True)
        et = _moving_meansq(we, W, centered=True)
        total += np.sum(np.abs(ep - et)) / (np.sum(np.abs(et)) + 1e-8)
    return float(total / len(windows))

def foes(gr_pred_db, gr_tgt_db):
    """Fidelity of Envelope Shape: normalised RMS error of the GR trajectory (Giannoulis et al.)."""
    err = np.sqrt(np.mean((gr_pred_db - gr_tgt_db) ** 2))
    return float(err / (np.sqrt(np.mean(gr_tgt_db ** 2)) + 1e-8))

def mr_nrmse(pred, wet, windows=MR_NRMSE_WINDOWS):
    """Mean normalised RMS-envelope error (== src.losses.multi_resolution_nrmse, cumsum-fast)."""
    total = 0.0
    for W in windows:
        rp = np.sqrt(_moving_meansq(pred * pred, W, centered=False))
        rt = np.sqrt(_moving_meansq(wet * wet, W, centered=False))
        total += np.sqrt(np.mean((rt - rp) ** 2)) / (np.sqrt(np.mean(rt * rt)) + 1e-8)
    return float(total / len(windows))

In [4]:
# ── Colour-stage + standard FFT metrics, batched across all detectors at once ──
_MRSTFT_RES = [(1024, 120, 600), (2048, 240, 1200), (512, 50, 240)]   # auraloss MRSTFT defaults
_HANN = {}
def _win(n):
    if n not in _HANN:
        _HANN[n] = torch.hann_window(n)
    return _HANN[n]

def _stft_mag_pow(x2d, n_fft, hop, win):
    """auraloss-style magnitude = sqrt(clamp(re^2 + im^2, 1e-8))."""
    st = torch.stft(x2d, n_fft=n_fft, hop_length=hop, win_length=win,
                    window=_win(win), return_complex=True)
    return torch.sqrt(torch.clamp(st.real ** 2 + st.imag ** 2, min=1e-8))

def _stft_mag(x2d, n_fft):
    """src.losses-style magnitude (hop = n_fft // 4, hann(n_fft))."""
    return torch.stft(x2d, n_fft=n_fft, hop_length=n_fft // 4,
                      window=_win(n_fft), return_complex=True).abs()

@torch.no_grad()
def fft_metrics_batched(preds2d: torch.Tensor, wet1d: torch.Tensor) -> dict:
    """preds2d: [D, T] (one row per detector), wet1d: [T]. Returns {col: list-of-D-floats}.

    Per-item reimplementations of auraloss MultiResolutionSTFTLoss and of
    src.losses.{m_stfte, multi_resolution_spectral_flux_error, edc}, validated to
    match the originals to <1e-3 / ~1e-7 respectively.
    """
    D = preds2d.shape[0]
    out = {}

    # MR-STFT (auraloss default = spectral-convergence + log-magnitude, mean over 3 resolutions)
    acc = torch.zeros(D)
    for n_fft, hop, win in _MRSTFT_RES:
        P = _stft_mag_pow(preds2d, n_fft, hop, win)
        T = _stft_mag_pow(wet1d.unsqueeze(0), n_fft, hop, win)[0]
        sc = torch.linalg.norm((T.unsqueeze(0) - P).reshape(D, -1), dim=1) / torch.linalg.norm(T.reshape(-1))
        lm = (torch.log(P) - torch.log(T.unsqueeze(0))).abs().mean(dim=(1, 2))
        acc += sc + lm
    out["MR-STFT (auraloss)"] = (acc / len(_MRSTFT_RES)).tolist()

    # src.losses MR-STFT (m_stfte) and spectral-flux (M_SF) over (512, 1024, 2048)
    ms = torch.zeros(D); sf = torch.zeros(D)
    for n_fft in STFT_SIZES:
        P = _stft_mag(preds2d, n_fft)
        T = _stft_mag(wet1d.unsqueeze(0), n_fft)[0]
        ms += (P - T.unsqueeze(0)).abs().sum(dim=(1, 2)) / (T.abs().sum() + 1e-8)
        pf = torch.diff(P, dim=-1); tf = torch.diff(T, dim=-1)
        sf += (tf.unsqueeze(0) - pf).abs().mean(dim=(1, 2)) / (tf.abs().mean() + 1e-8)
    out["MR-STFT"] = (ms / len(STFT_SIZES)).tolist()
    out["M_SF"]    = (sf / len(STFT_SIZES)).tolist()

    # EDC (energy-decay-curve error, dB)
    p = preds2d; t = wet1d.unsqueeze(0)
    pe = torch.flip(torch.cumsum(torch.flip(p ** 2, dims=(-1,)), dim=-1), dims=(-1,))
    te = torch.flip(torch.cumsum(torch.flip(t ** 2, dims=(-1,)), dim=-1), dims=(-1,))
    pe = pe / (pe[..., :1] + 1e-8); te = te / (te[..., :1] + 1e-8)
    out["EDC"] = ((10 * torch.log10(te.clamp(min=1e-8)) - 10 * torch.log10(pe.clamp(min=1e-8)))
                  .abs().mean(dim=-1)).tolist()
    return out

In [5]:
# ── Per-chunk and per-pair drivers ──
def chunk_metrics(dry: np.ndarray, wet: np.ndarray) -> dict:
    """All metrics for one chunk, every detector. -> {label: {col: float}}."""
    gr_tgt = np.clip(reference_gr_db(dry, wet), GR_DB_MIN, GR_DB_MAX)
    wet_aw = lfilter(AW_B, AW_A, wet)

    preds, rows = [], {}
    for label in DETECTORS:
        gr = detector_gr_db(label, dry, wet)
        pred = dry * to_amplitude(gr)                       # reconstruction (unclamped GR)
        preds.append(pred)
        gr_c = np.clip(gr, GR_DB_MIN, GR_DB_MAX)
        rows[label] = {
            "GR MAE (dB)": float(np.mean(np.abs(gr_c - gr_tgt))),
            "MR-STE":      mr_ste(pred, wet),
            "FoES":        foes(gr_c, gr_tgt),
            "ESR":         _esr(pred, wet),
            "ESR (A-wt)":  _esr(lfilter(AW_B, AW_A, pred), wet_aw),
            "MAE (L1)":    float(np.mean(np.abs(pred - wet))),
            "MSE (L2)":    float(np.mean((pred - wet) ** 2)),
            "M_NRMSE":     mr_nrmse(pred, wet),
        }

    fm = fft_metrics_batched(torch.from_numpy(np.stack(preds)).float(),
                             torch.from_numpy(wet).float())
    for i, label in enumerate(DETECTORS):
        for col in ("MR-STFT (auraloss)", "MR-STFT", "M_SF", "EDC"):
            rows[label][col] = fm[col][i]
    return rows

def pair_paths(song, setting):
    return (os.path.join(DATA_ROOT, "processed_normalized", f"{song}_UnmasteredWAV.wav"),
            os.path.join(DATA_ROOT, "processed_ground_truth", setting, f"{song}-exported.wav"))

def stream_pair(song, setting):
    """Frame-weighted metric sums for one (song, setting) pair (streamed, bounded memory)."""
    dry_p, wet_p = pair_paths(song, setting)
    total = _pair_num_frames(dry_p, wet_p, SAMPLE_RATE)
    acc = {l: {c: 0.0 for c in METRIC_COLS} for l in DETECTORS}
    n_frames = 0
    for o in range(0, total, CHUNK_FRAMES):
        d_t, w_t = _read_dry_wet_segment(dry_p, wet_p, o, min(o + CHUNK_FRAMES, total), SAMPLE_RATE)
        L = d_t.shape[-1] - (d_t.shape[-1] % STEP)
        if L < STEP:
            break
        dry = d_t.squeeze(0).numpy().astype(np.float64)[:L]
        wet = w_t.squeeze(0).numpy().astype(np.float64)[:L]
        cm = chunk_metrics(dry, wet)
        for l in DETECTORS:
            for c in METRIC_COLS:
                acc[l][c] += cm[l][c] * L
        n_frames += L
        del d_t, w_t, dry, wet, cm
    return acc, n_frames

## Discover every (song, setting) pair in the downloaded dataset


In [6]:
gt_root  = os.path.join(DATA_ROOT, "processed_ground_truth")
settings = sorted(d for d in os.listdir(gt_root) if d.startswith("threshold_"))

PAIRS = []
for setting in settings:
    for dry_path, wet_path in same_setting_pairs(DATA_ROOT, setting):
        PAIRS.append({"song": _song_from_dry_path(dry_path), "setting": setting})
PAIRS = sorted(PAIRS, key=lambda p: (p["setting"], p["song"]))

print(f"settings: {len(settings)}   (song, setting) pairs: {len(PAIRS)}")

settings: 10   (song, setting) pairs: 100


## Dataset stats

Cheap (header-only via `soundfile.info`, no decode) — the overall length of the set, the
control-parameter space, and the number of distinct sounds.

- A **song** is a distinct source audio; a **setting** is a `(threshold, attack, release, ratio)`
  combo. The same song is re-used across all settings, so **unique source material** counts each song
  once, while **total processed audio** counts every `(song, setting)` pair — the latter matches the
  full run's reported minutes further down.
- The 10 settings are a **sampled subset** of the 4-parameter grid (the values listed per knob are the
  union seen across those 10 settings, not a full Cartesian product).

In [ ]:
import soundfile as sf
import re

def _dur_min(path):
    info = sf.info(path)                                   # header-only, no decode
    return info.frames / info.samplerate / 60.0

songs = sorted({p["song"] for p in PAIRS})

# Parse the 4 control params out of each setting name
def _parse_setting(s):
    m = re.match(r"threshold_(-?[\d.]+)_attack_([\d.]+)_release_([\d.]+)_ratio_([\d.]+)", s)
    keys = ("threshold", "attack", "release", "ratio")
    return dict(zip(keys, m.groups())) if m else {}

param_vals = {
    k: sorted({_parse_setting(s)[k] for s in settings if _parse_setting(s)}, key=float)
    for k in ("threshold", "attack", "release", "ratio")
}

# Per-song breakdown. A song's dry source is re-used across all its settings, so
# "source_len_min" counts the song once while "processed_min" counts every pair.
by_song = {}
for p in PAIRS:
    by_song.setdefault(p["song"], []).append(p)

rows, total_processed = [], 0.0
for s in sorted(by_song):
    plist = by_song[s]
    src_min  = _dur_min(pair_paths(plist[0]["song"], plist[0]["setting"])[0])
    proc_min = sum(_dur_min(pair_paths(q["song"], q["setting"])[1]) for q in plist)
    total_processed += proc_min
    rows.append({
        "song": s,
        "n_settings":     len({q["setting"] for q in plist}),
        "n_pairs":        len(plist),
        "source_len_min": round(src_min, 2),
        "processed_min":  round(proc_min, 1),
    })
stats_df = pd.DataFrame(rows)
unique_src = stats_df["source_len_min"].sum()

print("Diff-SSL-G-Comp — dataset stats")
print(f"  distinct sounds (songs)          : {len(songs)}")
print(f"  control params (knobs)           : 4")
for k in ("threshold", "attack", "release", "ratio"):
    print(f"      - {k:9s}: {param_vals[k]}")
print(f"  distinct settings (sampled grid) : {len(settings)}")
print(f"  (song, setting) pairs            : {len(PAIRS)}")
print(f"  unique source material           : {unique_src:.1f} min  ({unique_src/60:.2f} h)")
print(f"  total processed audio (all pairs): {total_processed:.1f} min  ({total_processed/60:.2f} h)")
stats_df

## Verify the fast engine reproduces `nablafx` / auraloss

On the first chunk of the first pair, compare the batched colour-stage MR-STFT and ESR against
`nablafx.evaluation.get_function("mrstft_loss")` (auraloss `MultiResolutionSTFTLoss`) and
`get_function("esr_loss")` (auraloss `ESRLoss`) called per detector.


In [7]:
_song, _setting = PAIRS[0]["song"], PAIRS[0]["setting"]
_dp, _wp = pair_paths(_song, _setting)
_d, _w = _read_dry_wet_segment(_dp, _wp, 0, CHUNK_FRAMES, SAMPLE_RATE)
_L = _d.shape[-1] - (_d.shape[-1] % STEP)
_dry = _d.squeeze(0).numpy().astype(np.float64)[:_L]
_wet = _w.squeeze(0).numpy().astype(np.float64)[:_L]

_preds = np.stack([_dry * to_amplitude(detector_gr_db(l, _dry, _wet)) for l in DETECTORS])
_fm = fft_metrics_batched(torch.from_numpy(_preds).float(), torch.from_numpy(_wet).float())

_nab_mrstft = get_function("mrstft_loss")     # auraloss MultiResolutionSTFTLoss
_nab_esr    = get_function("esr_loss")        # auraloss ESRLoss
_wt = torch.from_numpy(_wet).float().view(1, 1, -1)

print(f"verifying on {_song} / {_setting}  (chunk 0)\n")
print(f"{'detector':12s} {'MRSTFT batched':>15s} {'MRSTFT nablafx':>15s} "
      f"{'ESR batched':>12s} {'ESR nablafx':>12s}")
for i, l in enumerate(DETECTORS):
    pt = torch.from_numpy(_preds[i]).float().view(1, 1, -1)
    nm, ne = float(_nab_mrstft(pt, _wt)), float(_nab_esr(pt, _wt))
    bm, be = _fm["MR-STFT (auraloss)"][i], _esr(_preds[i], _wet)
    assert abs(nm - bm) < 1e-3, (l, nm, bm)
    assert abs(ne - be) < 1e-4, (l, ne, be)
    print(f"{l:12s} {bm:15.6f} {nm:15.6f} {be:12.6f} {ne:12.6f}")
print("\nOK — batched fast path matches nablafx / auraloss.")

verifying on Air / threshold_-12_attack_10_release_0.4_ratio_10  (chunk 0)

detector      MRSTFT batched  MRSTFT nablafx  ESR batched  ESR nablafx
Hilbert             0.598411        0.598423     0.004220     0.004220
RMS 64              0.067423        0.067422     0.005559     0.005559
RMS 256             0.051751        0.051752     0.005766     0.005766
RMS 1024            0.057435        0.057434     0.006024     0.006024
RMS 4096            0.087238        0.087238     0.007521     0.007521
Simple diff         1.327487        1.327487     0.000366     0.000366

OK — batched fast path matches nablafx / auraloss.


## Run over the whole dataset

Overall, **duration-weighted** metrics only. ~20 s/pair → ~30 min for the full 100-pair set
(progress bar below). Lower the iteration count with `MAX_EVAL_PAIRS` for a smoke test.


In [8]:
pairs_to_eval = PAIRS if MAX_EVAL_PAIRS is None else PAIRS[:MAX_EVAL_PAIRS]

overall = {l: {c: 0.0 for c in METRIC_COLS} for l in DETECTORS}
total_frames = 0
for pair in tqdm(pairs_to_eval, desc="pairs", unit="pair"):
    acc, n = stream_pair(pair["song"], pair["setting"])
    for l in DETECTORS:
        for c in METRIC_COLS:
            overall[l][c] += acc[l][c]
    total_frames += n
    gc.collect()

print(f"\npairs evaluated: {len(pairs_to_eval)}   audio: {total_frames / SAMPLE_RATE / 60:.1f} min")

pairs: 100%|██████████| 100/100 [1:03:17<00:00, 37.97s/pair]


pairs evaluated: 100   audio: 440.6 min


## Overall results


In [9]:
overall_df = pd.DataFrame(
    [{"Detector": l, **{c: overall[l][c] / total_frames for c in METRIC_COLS}} for l in DETECTORS]
)
overall_df.to_csv(OUT_CSV, index=False)
print(f"saved -> {OUT_CSV}")

# Grouped view: GR-stage | colour-stage | standard
overall_df.set_index("Detector")[METRIC_COLS].round(6)

saved -> /Volumes/Saola's Drive/AllCode/thesis/Virtual-Analogue-Compressor-Modelling/01b_dset_anal/envelope_metrics_diffssl.csv


,GR MAE (dB),MR-STE,FoES,ESR,MR-STFT (auraloss),ESR (A-wt),MAE (L1),MSE (L2),MR-STFT,EDC,M_NRMSE,M_SF
Detector,,,,,,,,,,,,
Hilbert,0.858297,0.007314,0.322018,0.021281,0.669390,0.052426,0.001133,0.000005,0.093317,0.017361,0.004356,0.259568
RMS 64,0.266718,0.001480,0.077851,0.028023,0.085720,0.049724,0.001522,0.000007,0.019581,0.011509,0.000883,0.061640
RMS 256,0.170762,0.004032,0.045246,0.028585,0.064047,0.048287,0.001579,0.000008,0.014056,0.011748,0.002368,0.044987
RMS 1024,0.131805,0.011214,0.033411,0.029279,0.069051,0.047864,0.001606,0.000008,0.015757,0.012924,0.008110,0.049640
RMS 4096,0.161694,0.030545,0.044442,0.030769,0.090765,0.048772,0.001650,0.000008,0.026101,0.016440,0.025673,0.063336
Simple diff,1.929149,0.001034,0.793984,0.011341,0.928704,0.040366,0.000128,0.000002,0.092407,0.013189,0.000612,0.226109
